# Milestone 1 crawler on Colab (CPU)
One machine crawls the WHOLE assigned set (`id_ranges.csv`, decision D8). Put `S2_API_KEY` in Colab Secrets (never in the notebook). Colab can disconnect: data stays on the local disk (SQLite does not like Drive), the backup cell copies it to Google Drive, and the same command resumes the crawl.

In [ ]:
!git clone https://github.com/GiaaPhuc/arxiv-pipeline.git
%cd arxiv-pipeline/src
!pip install -q -r requirements.txt

In [ ]:
import os

from google.colab import drive, userdata

drive.mount("/content/drive")  # only used for backup / restore
os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
os.environ["CRAWLER_CONTACT"] = "your-email@example.com"  # shown in the User-Agent
os.environ["CRAWLER_DATA_DIR"] = "/content/data"

In [ ]:
# Plan only: expands the ranges, prints the id count, touches no network.
!python run_crawl.py --ranges id_ranges.csv --dry-run

In [ ]:
# Restore after a disconnect, BEFORE resuming:
# !rsync -a /content/drive/MyDrive/arxiv-data/ /content/data/

# Pilot first (30 ids), then the full run. Re-run the SAME command to resume.
!python run_crawl.py --ranges id_ranges.csv --limit 30
# !python run_crawl.py --ranges id_ranges.csv

In [ ]:
# Backup (run now and then; copies the manifest and every paper):
!rsync -a /content/data/ /content/drive/MyDrive/arxiv-data/

# After the crawl: check invariants, then build the tables and the report statistics.
!python validate_corpus.py --data-dir $CRAWLER_DATA_DIR
!python export_data.py --data-dir $CRAWLER_DATA_DIR